# Phase 5B - Alumina digestion efficiency

Regression on the 55 lab observations from dataset `78sbdmdbv6` (Li, Ga, Sc dissolution in roasted high-sulfur bauxite). Target is `digestion_efficiency_pct`, which ranges from 60.6% to 93.1%.

With only 55 rows this sticks to simple regularized models, leaves out red-mud features (they leak the target) and uses grouped validation. Lab data only, not an industrial predictor.

## 1. Setup & Data Loading

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    cross_val_score, LeaveOneGroupOut, GroupKFold, cross_val_predict
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
import warnings
warnings.filterwarnings('ignore')

print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
import sklearn; print(f"Scikit-learn: {sklearn.__version__}")

SEED = 42
np.random.seed(SEED)

In [ ]:
# Upload phase5_colab_dataset.csv and phase5_validation_groups.csv
from google.colab import files
print("Upload phase5_colab_dataset.csv:")
uploaded = files.upload()

In [ ]:
df = pd.read_csv('phase5_colab_dataset.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nBlocks: {df['block'].value_counts().to_dict()}")
print(f"\nTarget stats:")
print(df['digestion_efficiency_pct'].describe())

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Target distribution
axes[0, 0].hist(df['digestion_efficiency_pct'], bins=15, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Digestion Efficiency (%)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Target Distribution')
axes[0, 0].axvline(df['digestion_efficiency_pct'].mean(), color='red', linestyle='--', label='Mean')
axes[0, 0].legend()

# Target by block
block_order = ['Roasting desulfurization', 'Lime', 'Na2Ok', 'Temperature', 'Time']
df.boxplot(column='digestion_efficiency_pct', by='block', ax=axes[0, 1], rot=30)
axes[0, 1].set_title('Target by Experimental Block')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('Digestion Efficiency (%)')
plt.suptitle('')

# Target by feed type
df.boxplot(column='digestion_efficiency_pct', by='feed_type', ax=axes[1, 0])
axes[1, 0].set_title('Target by Feed Type')
axes[1, 0].set_xlabel('')
axes[1, 0].set_ylabel('Digestion Efficiency (%)')
plt.suptitle('')

# A/S ratio vs target
colors = {'Roasted': 'tab:orange', 'Unroasted': 'tab:blue'}
for ft, grp in df.groupby('feed_type'):
    axes[1, 1].scatter(grp['bauxite_as_ratio'], grp['digestion_efficiency_pct'],
                       label=ft, c=colors.get(ft), alpha=0.7, s=50)
axes[1, 1].set_xlabel('A/S Ratio')
axes[1, 1].set_ylabel('Digestion Efficiency (%)')
axes[1, 1].set_title('A/S Ratio vs Target')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Factor value vs target by block
fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=True)
for i, (block_name, grp) in enumerate(df.groupby('block')):
    for ft, subgrp in grp.groupby('feed_type'):
        axes[i].scatter(subgrp['factor_1_value'], subgrp['digestion_efficiency_pct'],
                        label=ft, alpha=0.8, s=60)
    axes[i].set_title(block_name, fontsize=10)
    axes[i].set_xlabel(grp['factor_1_name'].iloc[0])
    if i == 0:
        axes[i].set_ylabel('Digestion Efficiency (%)')
    axes[i].legend(fontsize=8)
plt.suptitle('Factor 1 Value vs Target by Block', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Feature engineering

Factor names differ between blocks, so each factor gets its own value column plus a presence flag.

In [ ]:
# Pivot factor values into explicit columns per factor type
factor_names = df['factor_1_name'].dropna().unique().tolist()
print(f"Unique factor_1 names: {factor_names}")
factor2_names = df['factor_2_name'].dropna().unique().tolist()
print(f"Unique factor_2 names: {factor2_names}")

all_factors = list(set(factor_names + factor2_names))
print(f"\nAll factor types: {all_factors}")

In [ ]:
def build_feature_matrix(df, include_as_ratio=False):
    """Build ML-ready feature matrix from the phase5 dataset."""
    feat = pd.DataFrame(index=df.index)

    # Feed type: binary encode
    feat['is_roasted'] = (df['feed_type'] == 'Roasted').astype(int)

    # Bauxite chemistry
    feat['al2o3'] = df['bauxite_al2o3_pct']
    feat['sio2'] = df['bauxite_sio2_pct']
    if include_as_ratio:
        feat['as_ratio'] = df['bauxite_as_ratio']

    # Pivot all factor values into dedicated columns, fill NaN with 0
    # and add a presence indicator for each
    all_factor_types = ['roasting_temperature', 'roasting_time', 'lime',
                        'Na2Ok', 'digestion_temperature', 'digestion_time']

    for ftype in all_factor_types:
        col_val = f'fv_{ftype}'
        col_present = f'fp_{ftype}'
        vals = pd.Series(np.nan, index=df.index)

        mask1 = df['factor_1_name'] == ftype
        vals[mask1] = df.loc[mask1, 'factor_1_value']

        mask2 = df['factor_2_name'] == ftype
        vals[mask2] = df.loc[mask2, 'factor_2_value']

        feat[col_present] = vals.notna().astype(int)
        feat[col_val] = vals.fillna(0.0)

    return feat


# Chemistry A: Al2O3 + SiO2 only
X_A = build_feature_matrix(df, include_as_ratio=False)
# Chemistry B: Al2O3 + SiO2 + A/S
X_B = build_feature_matrix(df, include_as_ratio=True)

y = df['digestion_efficiency_pct'].values

print(f"Chemistry A features ({X_A.shape[1]}): {list(X_A.columns)}")
print(f"Chemistry B features ({X_B.shape[1]}): {list(X_B.columns)}")
print(f"\nTarget shape: {y.shape}")
X_A.head()

In [ ]:
# Correlation heatmap
corr_df = X_B.copy()
corr_df['target'] = y
plt.figure(figsize=(14, 10))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Feature Correlation Matrix (Chemistry B)')
plt.tight_layout()
plt.show()

## 4. Validation

Leave-one-block-out, GroupKFold by sample_id (the main one), GroupKFold by composition, and a random 5-fold split for comparison only.

In [ ]:
# Prepare group arrays
groups_block = df['group_block'].values
groups_sample = df['group_sample_id'].values
groups_composition = df['group_composition_exact'].values
groups_feed_factor = df['group_feed_type_factor'].values

print("Validation group sizes:")
print(f"  Blocks: {df['group_block'].nunique()} groups")
print(f"  Sample IDs: {df['group_sample_id'].nunique()} groups")
print(f"  Compositions: {df['group_composition_exact'].nunique()} groups")
print(f"  Feed+Factor: {df['group_feed_type_factor'].nunique()} groups")

In [ ]:
def evaluate_model(model, X, y, cv_scheme, scheme_name):
    """Run cross-validation and return metrics dict."""
    y_pred = cross_val_predict(model, X, y, cv=cv_scheme)
    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)
    return {
        'scheme': scheme_name,
        'n': len(y),
        'MAE': round(mae, 3),
        'RMSE': round(rmse, 3),
        'R2': round(r2, 3)
    }


def run_all_validations(model, X, y, model_name):
    """Evaluate model under all validation schemes."""
    results = []

    # Leave-One-Block-Out
    logo = LeaveOneGroupOut()
    results.append(evaluate_model(model, X, y, logo.split(X, y, groups_block),
                                  'Leave-One-Block-Out'))

    # Group K-Fold by sample_id
    n_sample_groups = df['group_sample_id'].nunique()
    gkf_sample = GroupKFold(n_splits=min(5, n_sample_groups))
    results.append(evaluate_model(model, X, y,
                                  gkf_sample.split(X, y, groups_sample),
                                  f'GroupKFold(sample_id, k={min(5, n_sample_groups)})'))

    # Group K-Fold by feed+factor
    n_ff_groups = df['group_feed_type_factor'].nunique()
    gkf_ff = GroupKFold(n_splits=min(5, n_ff_groups))
    results.append(evaluate_model(model, X, y,
                                  gkf_ff.split(X, y, groups_feed_factor),
                                  f'GroupKFold(feed+factor, k={min(5, n_ff_groups)})'))

    # Random 5-fold (comparison only)
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    results.append(evaluate_model(model, X, y, kf, 'Random 5-Fold (ref only)'))

    res_df = pd.DataFrame(results)
    res_df.insert(0, 'model', model_name)
    return res_df

## 5. Models

Mean baseline, Ridge, Lasso, then random forest and gradient boosting. Each one is run on Chemistry A and B under every validation scheme.

In [ ]:
models = {
    'Mean Baseline': DummyRegressor(strategy='mean'),
    'Ridge (alpha=1.0)': Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=1.0, random_state=SEED))
    ]),
    'Ridge (alpha=10.0)': Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=10.0, random_state=SEED))
    ]),
    'Lasso (alpha=0.5)': Pipeline([
        ('scaler', StandardScaler()),
        ('lasso', Lasso(alpha=0.5, random_state=SEED, max_iter=5000))
    ]),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, max_depth=4, min_samples_leaf=3,
        random_state=SEED
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=50, max_depth=2, learning_rate=0.1,
        min_samples_leaf=3, random_state=SEED
    ),
}

In [ ]:
# Chemistry A evaluation
print("=" * 80)
print("CHEMISTRY A: Al2O3 + SiO2 (no A/S ratio)")
print("=" * 80)

all_results_A = []
for name, model in models.items():
    res = run_all_validations(model, X_A.values, y, name)
    all_results_A.append(res)

results_A = pd.concat(all_results_A, ignore_index=True)
print(results_A.to_string(index=False))

In [ ]:
# Chemistry B evaluation
print("=" * 80)
print("CHEMISTRY B: Al2O3 + SiO2 + A/S ratio")
print("=" * 80)

all_results_B = []
for name, model in models.items():
    res = run_all_validations(model, X_B.values, y, name)
    all_results_B.append(res)

results_B = pd.concat(all_results_B, ignore_index=True)
print(results_B.to_string(index=False))

## 6. Results Comparison

In [ ]:
# Compare Chemistry A vs B using GroupKFold(sample_id) as primary
primary_scheme = results_A['scheme'].unique()[1]  # GroupKFold sample_id

comp_A = results_A[results_A['scheme'] == primary_scheme][['model', 'MAE', 'RMSE', 'R2']].copy()
comp_A.columns = ['model', 'MAE_A', 'RMSE_A', 'R2_A']
comp_B = results_B[results_B['scheme'] == primary_scheme][['model', 'MAE', 'RMSE', 'R2']].copy()
comp_B.columns = ['model', 'MAE_B', 'RMSE_B', 'R2_B']

comparison = comp_A.merge(comp_B, on='model')
print(f"Primary validation: {primary_scheme}")
print()
print(comparison.to_string(index=False))

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

primary_A = results_A[results_A['scheme'] == primary_scheme]
primary_B = results_B[results_B['scheme'] == primary_scheme]

x_pos = np.arange(len(primary_A))
w = 0.35

for idx, (metric, title) in enumerate([('MAE', 'MAE (lower is better)'),
                                        ('RMSE', 'RMSE (lower is better)'),
                                        ('R2', 'R² (higher is better)')]):
    axes[idx].bar(x_pos - w/2, primary_A[metric].values, w, label='Chemistry A', alpha=0.8)
    axes[idx].bar(x_pos + w/2, primary_B[metric].values, w, label='Chemistry B', alpha=0.8)
    axes[idx].set_xticks(x_pos)
    axes[idx].set_xticklabels(primary_A['model'].values, rotation=45, ha='right', fontsize=8)
    axes[idx].set_title(title)
    axes[idx].legend(fontsize=8)

plt.suptitle(f'Model Comparison — {primary_scheme}', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Residual Analysis (Best Model)

In [ ]:
# Best model for residual analysis
best_name = 'Ridge (alpha=1.0)'
best_model = models[best_name]
best_X = X_A.values  # switch to X_B if Chemistry B wins

# Get cross-val predictions for residual analysis
n_sample_groups = df['group_sample_id'].nunique()
gkf = GroupKFold(n_splits=min(5, n_sample_groups))
y_pred_cv = cross_val_predict(best_model, best_X, y, cv=gkf.split(best_X, y, groups_sample))
residuals = y - y_pred_cv

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Predicted vs actual
axes[0, 0].scatter(y, y_pred_cv, c=pd.Categorical(df['block']).codes, cmap='tab10', alpha=0.7, s=50)
axes[0, 0].plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=1)
axes[0, 0].set_xlabel('Actual Digestion Efficiency (%)')
axes[0, 0].set_ylabel('Predicted Digestion Efficiency (%)')
axes[0, 0].set_title(f'{best_name} — Predicted vs Actual')

# Residuals vs predicted
axes[0, 1].scatter(y_pred_cv, residuals, alpha=0.7, s=50)
axes[0, 1].axhline(0, color='k', linestyle='--', lw=1)
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('Residual')
axes[0, 1].set_title('Residuals vs Predicted')

# Residuals histogram
axes[1, 0].hist(residuals, bins=12, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residual')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Residual Distribution')

# Residuals by block
res_df = pd.DataFrame({'block': df['block'], 'residual': residuals})
res_df.boxplot(column='residual', by='block', ax=axes[1, 1], rot=30)
axes[1, 1].set_title('Residuals by Block')
axes[1, 1].set_xlabel('')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f"\nResidual summary:")
print(f"  Mean: {residuals.mean():.3f}")
print(f"  Std:  {residuals.std():.3f}")
print(f"  Max absolute: {np.abs(residuals).max():.3f}")

## 8. Leave-One-Block-Out Diagnostic

In [ ]:
# Per-block LOBO diagnostic
logo = LeaveOneGroupOut()
block_results = []

for train_idx, test_idx in logo.split(best_X, y, groups_block):
    block_name = df.iloc[test_idx]['block'].iloc[0]
    n_test = len(test_idx)

    model_clone = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=1.0, random_state=SEED))
    ])
    model_clone.fit(best_X[train_idx], y[train_idx])
    y_pred_block = model_clone.predict(best_X[test_idx])

    block_results.append({
        'held_out_block': block_name,
        'n_test': n_test,
        'n_train': len(train_idx),
        'MAE': round(mean_absolute_error(y[test_idx], y_pred_block), 3),
        'RMSE': round(np.sqrt(mean_squared_error(y[test_idx], y_pred_block)), 3),
        'R2': round(r2_score(y[test_idx], y_pred_block), 3),
    })

lobo_df = pd.DataFrame(block_results)
print("Leave-One-Block-Out Diagnostic:")
print(lobo_df.to_string(index=False))

## 9. Final Model & Export

In [ ]:
# Train final model on all data
final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0, random_state=SEED))
])
final_model.fit(best_X, y)

# Feature importance via coefficients
feature_names = X_A.columns.tolist()
coeffs = final_model.named_steps['ridge'].coef_
coeff_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coeffs,
    'abs_coefficient': np.abs(coeffs)
}).sort_values('abs_coefficient', ascending=False)

print("Feature coefficients (standardized):")
print(coeff_df.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.barh(coeff_df['feature'], coeff_df['coefficient'])
plt.xlabel('Standardized Coefficient')
plt.title(f'{best_name} — Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Export trained model
import pickle

model_artifact = {
    'model': final_model,
    'feature_names': feature_names,
    'chemistry': 'A',
    'target': 'digestion_efficiency_pct',
    'training_n': len(y),
    'seed': SEED,
    'scope': 'proof-of-concept, laboratory data only',
}

with open('alumina_digestion_model.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)

print("Model saved to alumina_digestion_model.pkl")

# Download
from google.colab import files
files.download('alumina_digestion_model.pkl')

In [ ]:
# Also save full results
results_A['chemistry'] = 'A'
results_B['chemistry'] = 'B'
all_results = pd.concat([results_A, results_B], ignore_index=True)
all_results.to_csv('phase5b_evaluation_results.csv', index=False)
print("Results saved to phase5b_evaluation_results.csv")
files.download('phase5b_evaluation_results.csv')

## 10. Summary

| Item | Value |
|---|---|
| Dataset | phase5_colab_dataset.csv |
| Observations | 55 |
| Target | digestion_efficiency_pct |
| Primary validation | GroupKFold by sample_id |
| Best model | Ridge (alpha=1.0) |

Inputs are Al2O3 and SiO2, roasted or not, and the process factors (roasting temp/time, lime, Na2Ok, digestion temp/time). Lab scale only; not validated for industrial use.